# Módulo 2 · Práctica: un buscador de palabras similares

Programas con NumPy el álgebra lineal con la que un LLM representa palabras y acabas con un **buscador de palabras similares** y un **resolutor de analogías** (*rey − hombre + mujer ≈ reina*).

**Datos.** Sin red no hay embeddings preentrenados: usamos 40 palabras en 16 dimensiones construidas a mano con rasgos («realeza», «género», «animal»…) y algo de ruido. Los reales tienen miles de dimensiones aprendidas y sin nombre; la geometría es la misma.

Completa las celdas `# TU CÓDIGO` y ejecuta la `# Comprobación` siguiente.

| Parte | Qué haces |
| --- | --- |
| 1 | Vectores y producto escalar |
| 2 | Norma y similitud coseno |
| 3 | Matrices como transformaciones |
| 4 | Producto matricial |
| 5 | La matriz de embeddings |
| 6 | El buscador de palabras similares |
| 7 | Analogías |
| 8 | Visualizar y adelanto de la atención |
| 9 | Retos |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)

## Parte 1 · Vectores y producto escalar

Un vector de $\mathbb{R}^d$ es una lista de $d$ números. El producto escalar multiplica componente a componente y suma:

$$u \cdot v = \sum_{i=1}^{d} u_i v_i$$

**Ejercicio 1.** Implementa `producto_escalar(u, v)` con un bucle `for` (sin `np.dot`).

In [ ]:
def producto_escalar(u, v):
    # TU CÓDIGO
    return None

print(producto_escalar([1, 2, 3], [4, 5, 6]))

In [ ]:
# Comprobación
assert producto_escalar([1, 2, 3], [4, 5, 6]) == 32, "(1,2,3)·(4,5,6) debe valer 32"
assert producto_escalar([1, 0], [0, 1]) == 0, "Vectores perpendiculares: producto 0"
r = np.random.default_rng(1).normal(size=(2, 50))
assert np.isclose(producto_escalar(r[0], r[1]), r[0] @ r[1]), "Debe coincidir con el operador @"
print("Correcto. En NumPy se escribe u @ v (o np.dot(u, v)).")

## Parte 2 · Norma y similitud coseno

$$\|u\| = \sqrt{u \cdot u}, \qquad \cos(u, v) = \frac{u \cdot v}{\|u\|\,\|v\|}$$

**Ejercicio 2.** Implementa `norma(u)` y `coseno(u, v)` usando `producto_escalar` o `@`.

In [ ]:
def norma(u):
    # TU CÓDIGO
    return None

def coseno(u, v):
    # TU CÓDIGO
    return None

# Ejemplo del documento (dimensiones: realeza, masculino, persona, comida)
rey_j    = np.array([2.,  1., 2., 0.])
reina_j  = np.array([2., -1., 2., 0.])
hombre_j = np.array([0.,  1., 2., 0.])
print(norma(rey_j), coseno(rey_j, reina_j), coseno(rey_j, hombre_j))

In [ ]:
# Comprobación
assert np.isclose(norma([3, 4]), 5), "La norma de (3, 4) es 5"
assert np.isclose(coseno(rey_j, reina_j), 7 / 9), "cos(rey, reina) = 7/9 ≈ 0,778"
assert np.isclose(coseno([1, 2], [2, 4]), 1), "Vectores paralelos: coseno 1"
assert np.isclose(coseno([1, 0], [-1, 0]), -1), "Vectores opuestos: coseno -1"
print("Correcto")

**Pregunta.** ¿Por qué `coseno([1, 2], [2, 4])` vale 1 aunque las longitudes difieran? ¿Qué descarta el coseno?

## Parte 3 · Matrices como transformaciones

Una matriz $W$ de $m \times n$ es una función lineal $x \mapsto Wx$ de $\mathbb{R}^n$ en $\mathbb{R}^m$. La matriz de rotación de ángulo $\theta$ es

$$R_\theta = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}$$

**Ejercicio 3.** Escribe `rotacion(theta)` (ángulo en radianes).

In [ ]:
def rotacion(theta):
    # TU CÓDIGO
    return None

print(rotacion(np.pi / 2) @ np.array([1., 0.]))

In [ ]:
# Comprobación
R90 = rotacion(np.pi / 2)
assert np.allclose(R90 @ [1, 0], [0, 1]), "Girar (1, 0) 90° da (0, 1)"
assert np.allclose(R90 @ [3, 1], [-1, 3]), "Girar (3, 1) 90° da (-1, 3)"
x = np.array([2., 5.])
assert np.isclose(norma(rotacion(0.7) @ x), norma(x)), "Una rotación conserva la norma"
print("Correcto")

In [ ]:
# Cada columna de la matriz es la imagen de un vector de la base
casa = np.array([[0, 0], [1, 0], [1, 1], [0.5, 1.5], [0, 1], [0, 0]]).T   # 2 x 6 puntos
transformaciones = {
    "original": np.eye(2),
    "rotación 45°": rotacion(np.pi / 4),
    "escalado (2, 0,5)": np.array([[2, 0], [0, 0.5]]),
    "cizalla": np.array([[1, 1], [0, 1]]),
}
fig, axs = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, (nombre, W) in zip(axs, transformaciones.items()):
    P = W @ casa
    ax.plot(casa[0], casa[1], ":", color="gray")
    ax.plot(P[0], P[1], "-o")
    ax.set_title(nombre); ax.set_aspect("equal"); ax.set_xlim(-1.5, 2.5); ax.set_ylim(-1, 2)
plt.show()

## Parte 4 · Producto matricial

$(AB)_{ij} = \sum_k A_{ik} B_{kj}$: fila $i$ de $A$ por columna $j$ de $B$. Requiere que las columnas de $A$ coincidan con las filas de $B$.

**Ejercicio 4.** Implementa `matmul(A, B)` con bucles y lanza un `ValueError` si las formas no encajan.

In [ ]:
def matmul(A, B):
    A, B = np.asarray(A, float), np.asarray(B, float)
    # TU CÓDIGO
    return None

A = np.array([[1, 0, 2], [0, 1, -1]])
B = np.array([[1, 2], [0, 1], [3, 0]])
print(matmul(A, B))

In [ ]:
# Comprobación
assert np.allclose(matmul(A, B), [[7, 2], [-3, 1]]), "AB debe ser [[7, 2], [-3, 1]]"
assert matmul(B, A).shape == (3, 3), "BA es 3x3: el orden importa"
try:
    matmul(A, A); raise AssertionError("Debería fallar: (2x3)(2x3) no encaja")
except ValueError:
    pass
S = np.array([[2, 0], [0, 0.5]])
assert not np.allclose(matmul(R90, S), matmul(S, R90)), "Rotar y escalar no conmuta"
assert np.allclose(matmul(R90, S) @ [3, 1], R90 @ (S @ [3, 1])), "(RS)x = R(Sx): composición"
print("Correcto. En NumPy: A @ B")

## Parte 5 · La matriz de embeddings

Cada fila de la tabla siguiente describe una palabra con rasgos. La convertimos en una matriz $E$ de $|V| \times d$ (40 × 16) y añadimos ruido gaussiano pequeño para que no sea perfecta, como ocurre con los embeddings aprendidos.

In [ ]:
DIMS = ["realeza", "género", "edad", "persona", "familia", "animal", "doméstico", "felino",
        "comida", "fruta", "elaborado", "lugar", "ciudad", "este", "norte", "tamaño"]
RASGOS = """
rey       realeza=1.5 género=0.5 edad=1 persona=1
reina     realeza=1.5 género=-0.5 edad=1 persona=1
príncipe  realeza=1.5 género=0.5 edad=-0.5 persona=1 familia=0.5
princesa  realeza=1.5 género=-0.5 edad=-0.5 persona=1 familia=0.5
hombre    género=0.5 edad=1 persona=1
mujer     género=-0.5 edad=1 persona=1
niño      género=0.5 edad=-1 persona=1
niña      género=-0.5 edad=-1 persona=1
chico     género=0.5 edad=-0.5 persona=1
chica     género=-0.5 edad=-0.5 persona=1
padre     género=0.5 edad=1 persona=1 familia=1
madre     género=-0.5 edad=1 persona=1 familia=1
hijo      género=0.5 edad=-0.5 persona=1 familia=1
hija      género=-0.5 edad=-0.5 persona=1 familia=1
hermano   género=0.5 persona=1 familia=1
hermana   género=-0.5 persona=1 familia=1
perro     animal=1 doméstico=1
gato      animal=1 doméstico=1 felino=1 tamaño=-0.5
león      animal=1 doméstico=-1 felino=1 género=0.5 tamaño=1 realeza=0.45
leona     animal=1 doméstico=-1 felino=1 género=-0.5 tamaño=1
caballo   animal=1 doméstico=1 género=0.5 tamaño=1
yegua     animal=1 doméstico=1 género=-0.5 tamaño=1
toro      animal=1 doméstico=1 género=0.5 tamaño=1.2
vaca      animal=1 doméstico=1 género=-0.5 tamaño=1.2
gallo     animal=1 doméstico=1 género=0.5 tamaño=-0.5
gallina   animal=1 doméstico=1 género=-0.5 tamaño=-0.5
manzana   comida=1 fruta=1
pera      comida=1 fruta=1
naranja   comida=1 fruta=1
plátano   comida=1 fruta=1
pan       comida=1 elaborado=1
queso     comida=1 elaborado=1 doméstico=0.4
España    lugar=1 ciudad=-1 este=-0.4 norte=0
Madrid    lugar=1 ciudad=1 este=-0.4 norte=0
Portugal  lugar=1 ciudad=-1 este=-0.9 norte=-0.2
Lisboa    lugar=1 ciudad=1 este=-0.9 norte=-0.2
Francia   lugar=1 ciudad=-1 este=0.2 norte=0.9
París     lugar=1 ciudad=1 este=0.2 norte=0.9
Italia    lugar=1 ciudad=-1 este=1.2 norte=0.2
Roma      lugar=1 ciudad=1 este=1.2 norte=0.2
"""

palabras, filas = [], []
for linea in RASGOS.strip().splitlines():
    p, *pares = linea.split()
    v = np.zeros(len(DIMS))
    for par in pares:
        k, x = par.split("=")
        v[DIMS.index(k)] = float(x)
    palabras.append(p); filas.append(v)
E = np.array(filas) + np.random.default_rng(0).normal(0, 0.1, (len(filas), len(DIMS)))
E = E.round(2)
idx = {p: i for i, p in enumerate(palabras)}
V, d = E.shape
print(f"|V| = {V} palabras, d = {d} dimensiones")
print("rey =", E[idx["rey"]])

Un LLM no recibe la palabra: recibe su **identificador** (un entero, módulo 1). La capa de embedding equivale a multiplicar un vector *one-hot* por $E$.

**Ejercicio 5.** Escribe `one_hot(i, V)` y comprueba que `one_hot(i, V) @ E` es la fila `i` de `E`. Después construye la matriz `X` (3 × 16) de la frase «reina gato Madrid» multiplicando una matriz de one-hots por `E`.

In [ ]:
def one_hot(i, V):
    # TU CÓDIGO
    return None

frase = ["reina", "gato", "Madrid"]
X = None   # TU CÓDIGO: matriz 3 x d

In [ ]:
# Comprobación
assert one_hot(2, 5).tolist() == [0, 0, 1, 0, 0], "one_hot(2, 5) = [0, 0, 1, 0, 0]"
assert np.allclose(one_hot(idx["rey"], V) @ E, E[idx["rey"]]), "one-hot @ E selecciona una fila"
assert X is not None and X.shape == (3, d), "X debe ser 3 x d"
assert np.allclose(X, E[[idx[p] for p in frase]]), "X debe coincidir con indexar E por filas"
print("Correcto. En la práctica nadie multiplica: se indexa E[ids], que es lo mismo y más barato.")

## Parte 6 · El buscador de palabras similares

Si normalizamos cada fila de $E$ a norma 1 (matriz $\hat E$), el coseno de la palabra $i$ con todas las demás es un único producto matriz-vector: $\hat E\,\hat e_i$.

**Ejercicio 6.** Calcula `En` (filas de norma 1) y escribe `mas_similares(palabra, k)` que devuelva una lista de `k` pares `(palabra, coseno)` ordenados de mayor a menor, **excluyendo** la propia palabra. Pista: `np.argsort(-s)`.

In [ ]:
En = None   # TU CÓDIGO

def mas_similares(palabra, k=5):
    # TU CÓDIGO
    return None

for w in ["rey", "perro", "manzana", "Madrid"]:
    print(w, "->", mas_similares(w, 4))

In [ ]:
# Comprobación
assert En is not None and np.allclose(np.linalg.norm(En, axis=1), 1), "Cada fila de En debe tener norma 1"
top = mas_similares("rey", 3)
assert len(top) == 3 and all(p != "rey" for p, _ in top), "Excluye la propia palabra"
assert top[0][0] == "reina" and abs(top[0][1] - 0.874) < 1e-3, "El vecino de rey es reina (0,874)"
assert mas_similares("manzana", 1)[0][0] == "pera", "El vecino de manzana es pera"
assert [c for _, c in top] == sorted([c for _, c in top], reverse=True), "Orden descendente"
print("Correcto")

**Pregunta.** Busca los vecinos de `león`. ¿Por qué tras `leona` aparece `príncipe`? Mira qué rasgos comparte con `rey`.

## Parte 7 · Analogías

«a es a b como ? es a c» se resuelve buscando la palabra más cercana (por coseno) a $q = e_a - e_b + e_c$, excluyendo $a$, $b$ y $c$. Para *rey − hombre + mujer*, la resta quita «masculino adulto» y la suma pone «femenino adulto».

**Ejercicio 7.** Escribe `analogia(a, b, c, k=1)` que devuelva los `k` mejores pares `(palabra, coseno)`.

In [ ]:
def analogia(a, b, c, k=1):
    # TU CÓDIGO
    return None

pruebas = [("rey", "hombre", "mujer"), ("padre", "hombre", "mujer"), ("Madrid", "España", "Francia"),
           ("París", "Francia", "Italia"), ("caballo", "toro", "vaca"), ("león", "rey", "reina")]
for a, b, c in pruebas:
    print(f"{a} - {b} + {c} = {analogia(a, b, c, 2)}")

In [ ]:
# Comprobación
r = analogia("rey", "hombre", "mujer", 2)
assert r[0][0] == "reina" and abs(r[0][1] - 0.924) < 1e-3, "rey - hombre + mujer = reina (0,924)"
assert analogia("Madrid", "España", "Francia")[0][0] == "París"
assert analogia("Lisboa", "Portugal", "España")[0][0] == "Madrid"
assert all(p not in ("rey", "hombre", "mujer") for p, _ in r), "Excluye las palabras de la consulta"
print("Correcto")

**Pregunta.** Repite `rey - hombre + mujer` **sin** excluir las palabras de la consulta. ¿Qué palabra queda segunda? (Sección 2.7 del documento.)

## Parte 8 · Visualizar y adelanto de la atención

Proyectamos sobre dos ejes con nombre: «género» y «realeza». Con embeddings reales no hay ejes con nombre y se usa PCA (módulo 3).

In [ ]:
personas = ["rey", "reina", "príncipe", "princesa", "hombre", "mujer", "niño", "niña", "padre", "madre"]
g, r_ = DIMS.index("género"), DIMS.index("realeza")
fig, ax = plt.subplots(figsize=(6, 5))
for p in personas:
    x, y = E[idx[p], g], E[idx[p], r_]
    ax.scatter(x, y); ax.annotate(p, (x, y), textcoords="offset points", xytext=(4, 4))
for a, b in [("hombre", "rey"), ("mujer", "reina")]:
    ax.annotate("", E[idx[b], [g, r_]], E[idx[a], [g, r_]], arrowprops=dict(arrowstyle="->", color="gray"))
ax.set_xlabel("género (+ masculino, − femenino)"); ax.set_ylabel("realeza")
ax.set_title("Las dos flechas hombre→rey y mujer→reina son casi paralelas")
plt.show()

In [ ]:
# En @ En.T (40 x 40) contiene TODOS los cosenos con un solo producto matricial
Sim = En @ En.T
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(Sim, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(V)); ax.set_xticklabels(palabras, rotation=90, fontsize=7)
ax.set_yticks(range(V)); ax.set_yticklabels(palabras, fontsize=7)
plt.colorbar(im); ax.set_title("Similitud coseno entre las 40 palabras"); plt.show()

La atención del módulo 13 hace algo parecido: $QK^\top$ compara cada token con todos los demás mediante productos escalares.

**Ejercicio 8 (adelanto de la atención).** Para la frase `X` de la parte 5 calcula la matriz de puntuaciones `S = X @ X.T / np.sqrt(d)` y aplica una softmax por filas para obtener `A` (cada fila suma 1).

In [ ]:
S = None   # TU CÓDIGO
A = None   # TU CÓDIGO
print(A)

In [ ]:
# Comprobación
assert S is not None and S.shape == (3, 3) and np.allclose(S, S.T), "S es 3x3 y simétrica"
assert np.allclose(A.sum(axis=1), 1) and (A > 0).all(), "Cada fila de A es una distribución"
assert (A.argmax(axis=1) == [0, 1, 2]).all(), "Cada palabra se atiende sobre todo a sí misma"
print("Correcto. En el módulo 13, Q y K serán X multiplicada por dos matrices aprendidas.")

## Parte 9 · Retos

1. **Alta dimensión.** Con 2.000 pares de vectores gaussianos en d = 2, 16, 768 y 12.288, compara la desviación típica del coseno con $1/\sqrt{d}$.
2. **Escalar frente a coseno.** Multiplica por 3 la fila de `pan` y busca con el producto escalar sin normalizar. ¿Qué cambia?
3. **Euclídea.** Busca con distancia euclídea sobre `En`: el orden coincide ($\|u-v\|^2 = 2 - 2\cos$).
4. **Tu palabra.** Añade «duque» y «duquesa». ¿Funciona `duque − hombre + mujer`?

---
### Soluciones de referencia
Inténtalo antes de mirar.

In [ ]:
# Ejercicio 1
#     total = 0
#     for a, b in zip(u, v):
#         total += a * b
#     return total
# Ejercicio 2
#     norma:  return np.sqrt(producto_escalar(u, u))
#     coseno: return producto_escalar(u, v) / (norma(u) * norma(v))
# Ejercicio 3
#     c, s = np.cos(theta), np.sin(theta)
#     return np.array([[c, -s], [s, c]])
# Ejercicio 4
#     if A.shape[1] != B.shape[0]:
#         raise ValueError(f"No encajan: {A.shape} y {B.shape}")
#     C = np.zeros((A.shape[0], B.shape[1]))
#     for i in range(A.shape[0]):
#         for j in range(B.shape[1]):
#             for k in range(A.shape[1]):
#                 C[i, j] += A[i, k] * B[k, j]
#     return C
# Ejercicio 5
#     v = np.zeros(V); v[i] = 1; return v
#     X = np.array([one_hot(idx[p], V) for p in frase]) @ E
# Ejercicio 6
# En = E / np.linalg.norm(E, axis=1, keepdims=True)
#     s = En @ En[idx[palabra]]
#     orden = [j for j in np.argsort(-s) if j != idx[palabra]]
#     return [(palabras[j], float(s[j])) for j in orden[:k]]
# Ejercicio 7
#     q = E[idx[a]] - E[idx[b]] + E[idx[c]]
#     s = En @ (q / np.linalg.norm(q))
#     orden = [j for j in np.argsort(-s) if palabras[j] not in (a, b, c)]
#     return [(palabras[j], float(s[j])) for j in orden[:k]]
# Ejercicio 8
# S = X @ X.T / np.sqrt(d)
# A = np.exp(S - S.max(axis=1, keepdims=True)); A = A / A.sum(axis=1, keepdims=True)